# Práctica 1 · Introducción al aprendizaje por refuerzo

**DIH DATalife** — Curso de Reinforcement Learning (RL) desde cero

Este cuaderno es la práctica guiada de la clase uno. Retoma el entorno de pasillo que programaste en clase y te propone dos ejercicios para ampliarlo y para comparar políticas.

**Antes de empezar**:

- Ejecuta las celdas en orden, de arriba abajo
- Cada ejercicio trae su propio andamiaje: código ya escrito con partes marcadas como `# TODO`
- Completa cada `TODO` y ejecuta la celda para comprobar tu solución
- Si te atascas, despliega la solución que hay justo debajo de cada ejercicio, pero **inténtalo primero por tu cuenta**

Este cuaderno funciona en Google Colab sin instalar nada: solo necesitas `numpy`, que ya viene preinstalado.


## Preparación: el entorno de la clase

Ejecuta esta celda para tener disponibles el `PasilloEntorno`, la `politica_aleatoria` y una función auxiliar (`contar_pasos_hasta_meta`) que usarás en el ejercicio dos. Es el mismo código que viste en la clase.


In [ ]:
import numpy as np


class PasilloEntorno:
    def __init__(self, longitud=5):
        self.longitud = longitud
        self.meta = longitud - 1
        self.estado = 0  # posición inicial del agente

    def reset(self):
        """Reinicia el entorno y devuelve el estado inicial."""
        self.estado = 0
        return self.estado

    def step(self, accion):
        """
        Ejecuta una acción en el entorno.
        accion: 0 = mover a la izquierda, 1 = mover a la derecha
        Devuelve: (nuevo_estado, recompensa, terminado)
        """
        if accion == 1:
            self.estado = min(self.estado + 1, self.longitud - 1)
        elif accion == 0:
            self.estado = max(self.estado - 1, 0)

        terminado = self.estado == self.meta
        recompensa = 10 if terminado else -1

        return self.estado, recompensa, terminado


def politica_aleatoria(estado, num_acciones=2):
    """Política estocástica: elige una acción al azar, sin usar el estado."""
    return np.random.randint(num_acciones)


def contar_pasos_hasta_meta(entorno, elegir_accion, max_pasos=50, semilla=None):
    """
    Ejecuta un episodio completo con la política que le pases en
    'elegir_accion' (una función que recibe el estado y devuelve una
    acción) y devuelve cuántos pasos tardó en llegar a la meta, o
    max_pasos si no llegó a tiempo. La usarás en el ejercicio dos.
    """
    if semilla is not None:
        np.random.seed(semilla)

    estado = entorno.reset()
    for paso in range(1, max_pasos + 1):
        accion = elegir_accion(estado)
        estado, recompensa, terminado = entorno.step(accion)
        if terminado:
            return paso
    return max_pasos


print("Entorno y funciones auxiliares listos.")

## Ejercicio 1 · Del pasillo a una cuadrícula con trampa

El pasillo de la clase solo tiene una dimensión: izquierda o derecha. En este ejercicio vas a ampliarlo a una **cuadrícula 2D pequeña** con una celda "trampa" que da recompensa negativa.

**Reglas del nuevo entorno `CuadriculaEntorno`**:

- El agente empieza en la esquina superior izquierda, en la posición (0, 0)
- La meta está en la esquina inferior derecha, en la posición (filas-1, columnas-1)
- Hay cuatro acciones posibles: 0 = arriba, 1 = abajo, 2 = izquierda, 3 = derecha
- Si el agente pisa la celda trampa, el episodio termina ahí mismo con recompensa -5 (no llega a la meta)
- Si el agente llega a la meta, el episodio termina con recompensa +10
- En cualquier otro paso, la recompensa es -1, igual que en el pasillo

**Pistas**:

- El estado ahora es una tupla `(fila, columna)`, en vez de un único número
- Para no salirte de la cuadrícula, usa `min` y `max` igual que hace `PasilloEntorno` con `self.longitud`
- Compara siempre `self.estado` (ya actualizado) contra `self.trampa` y contra `self.meta` para decidir la recompensa y si el episodio termina

Completa los dos `TODO` de la celda siguiente y ejecútala para comprobar tu solución con un episodio de ejemplo.


In [ ]:
class CuadriculaEntorno:
    """
    Cuadrícula de filas x columnas. El agente empieza en (0, 0) y la
    meta está en (filas-1, columnas-1). Hay una celda "trampa": si el
    agente cae en ella, recibe una recompensa negativa y el episodio
    termina ahí mismo, sin llegar a la meta.

    Acciones: 0 = arriba, 1 = abajo, 2 = izquierda, 3 = derecha
    """

    def __init__(self, filas=3, columnas=3, trampa=(1, 1)):
        self.filas = filas
        self.columnas = columnas
        self.meta = (filas - 1, columnas - 1)
        self.trampa = trampa
        self.estado = (0, 0)

    def reset(self):
        self.estado = (0, 0)
        return self.estado

    def step(self, accion):
        fila, columna = self.estado

        # TODO 1: actualiza 'fila' y 'columna' según la acción.
        # Usa min/max para no salirte de la cuadrícula, igual que
        # hace PasilloEntorno con self.longitud.
        #   0 = arriba    -> fila - 1
        #   1 = abajo     -> fila + 1
        #   2 = izquierda -> columna - 1
        #   3 = derecha   -> columna + 1

        self.estado = (fila, columna)

        # TODO 2: calcula 'recompensa' y 'terminado' según el nuevo
        # self.estado:
        #   - si self.estado es self.trampa -> recompensa -5, terminado True
        #   - si self.estado es self.meta   -> recompensa +10, terminado True
        #   - en cualquier otro caso        -> recompensa -1, terminado False
        recompensa = None
        terminado = None

        return self.estado, recompensa, terminado


def politica_aleatoria_cuadricula(estado, num_acciones=4):
    return np.random.randint(num_acciones)


def ejecutar_episodio_cuadricula(entorno, max_pasos=30, semilla=None):
    """Igual que ejecutar_episodio del pasillo, adaptado a la cuadrícula."""
    if semilla is not None:
        np.random.seed(semilla)

    estado = entorno.reset()
    recompensa_acumulada = 0

    for paso in range(1, max_pasos + 1):
        accion = politica_aleatoria_cuadricula(estado)
        nuevo_estado, recompensa, terminado = entorno.step(accion)
        recompensa_acumulada += recompensa

        print(f"Paso {paso}: {estado} -> acción={accion} -> {nuevo_estado}, recompensa={recompensa}")

        estado = nuevo_estado

        if terminado:
            if estado == entorno.trampa:
                print(f"\nCayó en la trampa en el paso {paso}")
            else:
                print(f"\n¡Meta alcanzada en el paso {paso}!")
            break

    print(f"Recompensa acumulada del episodio: {recompensa_acumulada}")
    return recompensa_acumulada


# Prueba tu código con un episodio de ejemplo
ejecutar_episodio_cuadricula(CuadriculaEntorno(), max_pasos=30, semilla=3)

<details>
<summary>💡 Ver solución</summary>

```python
class CuadriculaEntorno:
    def __init__(self, filas=3, columnas=3, trampa=(1, 1)):
        self.filas = filas
        self.columnas = columnas
        self.meta = (filas - 1, columnas - 1)
        self.trampa = trampa
        self.estado = (0, 0)

    def reset(self):
        self.estado = (0, 0)
        return self.estado

    def step(self, accion):
        fila, columna = self.estado

        # TODO 1 resuelto: aplicamos el desplazamiento según la acción,
        # recortando con min/max para no salirnos de la cuadrícula
        if accion == 0:
            fila = max(fila - 1, 0)
        elif accion == 1:
            fila = min(fila + 1, self.filas - 1)
        elif accion == 2:
            columna = max(columna - 1, 0)
        elif accion == 3:
            columna = min(columna + 1, self.columnas - 1)

        self.estado = (fila, columna)

        # TODO 2 resuelto: comparamos el nuevo estado con la trampa y
        # con la meta para decidir recompensa y si el episodio termina
        if self.estado == self.trampa:
            recompensa = -5
            terminado = True
        elif self.estado == self.meta:
            recompensa = 10
            terminado = True
        else:
            recompensa = -1
            terminado = False

        return self.estado, recompensa, terminado
```

**Por qué funciona así**: el patrón es el mismo que en `PasilloEntorno`, solo que ahora tienes dos coordenadas en vez de una. Comprueba primero la trampa y después la meta (en ese orden) porque, en una cuadrícula más grande, podrían coincidir escenarios donde el orden de las comprobaciones importa; aquí, con `trampa` y `meta` en celdas distintas, el orden no cambia el resultado, pero es buena costumbre mantenerlo explícito.
</details>


## Ejercicio 2 · Política codiciosa contra política aleatoria

En clase viste solo la política aleatoria: no usa ninguna información del estado, elige cada acción al azar. En este ejercicio vas a implementar una **política codiciosa** (*greedy*): una política determinista y muy simple que, como conoce de antemano dónde está la meta, siempre se mueve hacia ella.

Vas a trabajar sobre el `PasilloEntorno` de la clase (no sobre la cuadrícula del ejercicio uno), para poder comparar ambas políticas en el mismo entorno.

**Pistas**:

- El estado del pasillo es un número entre cero y la posición de la meta
- Si el estado está antes de la meta, la acción correcta es moverse a la derecha (1)
- Si el estado está después de la meta, la acción correcta es moverse a la izquierda (0); en este entorno concreto no debería llegar a ocurrir, pero conviene contemplarlo
- Usa `contar_pasos_hasta_meta`, ya definida en la celda de preparación, para medir cuántos pasos tarda cada política en llegar a la meta

Completa el `TODO` de `politica_codiciosa` y ejecuta la celda para comparar los pasos medios de veinte episodios con cada política.


In [ ]:
entorno_pasillo = PasilloEntorno()

# Referencia: cuántos pasos tarda la política aleatoria del capítulo,
# medidos en veinte episodios
pasos_aleatoria = [
    contar_pasos_hasta_meta(entorno_pasillo, politica_aleatoria, max_pasos=50)
    for _ in range(20)
]


def politica_codiciosa(estado):
    """
    Política determinista: siempre se mueve hacia la meta. Como
    conoces la posición de la meta de antemano, no hace falta
    explorar para decidir la acción.
    """
    meta = entorno_pasillo.meta

    # TODO: devuelve 1 (derecha) si 'estado' está antes de la meta,
    # o 0 (izquierda) si 'estado' está después de la meta.


pasos_codiciosa = [
    contar_pasos_hasta_meta(entorno_pasillo, politica_codiciosa, max_pasos=50)
    for _ in range(20)
]

print("Media de pasos - política aleatoria:", np.mean(pasos_aleatoria))
print("Media de pasos - política codiciosa:", np.mean(pasos_codiciosa))

<details>
<summary>💡 Ver solución</summary>

```python
def politica_codiciosa(estado):
    meta = entorno_pasillo.meta

    # TODO resuelto: comparamos el estado con la meta y elegimos la
    # acción que acerca al agente
    if estado < meta:
        return 1  # derecha
    elif estado > meta:
        return 0  # izquierda
    else:
        return 1  # por seguridad, aunque no debería llamarse ya en la meta
```

**Qué deberías observar**: la política aleatoria tarda una media de bastantes pasos, con mucha variación de un episodio a otro (a veces llega rápido, a veces se queda dando vueltas). La política codiciosa, al conocer de antemano dónde está la meta, tarda siempre el mismo número de pasos: la distancia inicial hasta la meta, sin ningún paso de más.

Esta comparación es justo el punto de partida de los próximos capítulos: la política codiciosa de aquí "hace trampa" porque ya conoce la posición de la meta de antemano. Un agente real casi nunca tiene esa información: por eso hace falta un algoritmo, como Q-learning, que la aprenda a partir de la experiencia en vez de dártela ya resuelta.
</details>


## Para seguir practicando

Si te ha sobrado tiempo, retoma las propuestas del manual:

- Añade una tercera acción, por ejemplo "quedarse quieto", tanto en `PasilloEntorno` como en `CuadriculaEntorno`
- Cambia el diseño de la recompensa (por ejemplo, 0 por paso en vez de -1) y observa si el agente sigue teniendo incentivo para llegar rápido a la meta
- Combina los dos ejercicios: prueba una política codiciosa sobre `CuadriculaEntorno` que evite la trampa a propósito

Guarda tus resultados: los volverás a usar como punto de comparación cuando, en próximas clases, sustituyas estas políticas por una que aprende de verdad, como Q-learning.
